In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns # iris(데이터프레임)가져오기
from sklearn import datasets # iris(X와 y가 분리되서 numpy) 가져오기
from sklearn.preprocessing import LabelEncoder # 라벨 인코더
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from matplotlib import pyplot as plt

```
- 데이터셋 생성 및 전처리
    X, y(라벨인코딩)분리 -> train_test_split 분리(8:2)
- 모델 구성(입력4, 출력3, layer : 4->64->128->50->30->3)
- 학습과정 설정 (loss='sparse_categorical_crossentropy')
- 학습
- 모델평가(가장 최선의 모델을 get)
- 모델 사용(저장/예측)
```
# 1. 기본적인 DNN
## 1. 데이터 셋 생성 및 전처리

In [5]:
# iris가져오기 방법 1
iris = sns.load_dataset('iris')
iris['species'].value_counts()

setosa        50
versicolor    50
virginica     50
Name: species, dtype: int64

In [12]:
iris_X = iris.iloc[:, :-1].values
labelEncoder = LabelEncoder()
iris_y = labelEncoder.fit_transform(iris.iloc[:, -1]) # 0:setosa/1:versicolor/2:virginica
iris_X.shape, iris_y.shape

((150, 4), (150,))

In [15]:
# iris가져오기 방법 2
iris = datasets.load_iris()
iris_X = iris.data
iris_y = iris.target
print(iris.target_names)
iris_X.shape, iris_y.shape

['setosa' 'versicolor' 'virginica']


((150, 4), (150,))

In [24]:
# 학습셋과 테스트셋 분리
train_X, test_X, train_y, test_y = train_test_split(iris_X, iris_y,
                                                   test_size=0.2,
                                                   stratify=iris_y # 층화추출
                                                   )
np.c_[pd.Series(iris_y).value_counts(normalize=True),
      pd.Series(train_y).value_counts(normalize=True),
      pd.Series(test_y).value_counts(normalize=True)]

array([[0.33333333, 0.33333333, 0.33333333],
       [0.33333333, 0.33333333, 0.33333333],
       [0.33333333, 0.33333333, 0.33333333]])

In [25]:
train_X.shape, train_y.shape, test_X.shape, test_y.shape

((120, 4), (120,), (30, 4), (30,))

## 2. 모델 구성 및 학습
- 4->64->128->50->30->3
- He초기화, 다양한 activation, 배치정규화, L2정규화, Dropout

In [29]:
model = Sequential([
    Input(4),
    Dense(units=64, activation='relu'),
    Dense(units=128, activation='relu'),
    Dense(units=50, activation='relu'),
    Dense(units=30, activation='relu'),
    Dense(units=3, activation='softmax'),
], 'sequential')
# model = Sequential(name='sequential')
# model.add(Input(4))
# model.add(Dense(units=64, activation='relu'))
# model.add(Dense(units=128, activation='relu'))
# model.add(Dense(units=50, activation='relu'))
# model.add(Dense(units=30, activation='relu'))
# model.add(Dense(units=3, activation='softmax'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_10 (Dense)            (None, 64)                320       
                                                                 
 dense_11 (Dense)            (None, 128)               8320      
                                                                 
 dense_12 (Dense)            (None, 50)                6450      
                                                                 
 dense_13 (Dense)            (None, 30)                1530      
                                                                 
 dense_14 (Dense)            (None, 3)                 93        
                                                                 
Total params: 16,713
Trainable params: 16,713
Non-trainable params: 0
_________________________________________________________________


In [30]:
%%time
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])
earlystopping = EarlyStopping(patience=40) # monitor='val_loss'가 기본값
hist = model.fit(train_X, train_y,
                epochs=200,
                validation_split=0.1,
                callbacks=[earlystopping],
                verbose=2)

Epoch 1/200
4/4 - 2s - loss: 1.0577 - accuracy: 0.3889 - val_loss: 0.9613 - val_accuracy: 0.3333 - 2s/epoch - 537ms/step
Epoch 2/200
4/4 - 0s - loss: 0.9377 - accuracy: 0.5648 - val_loss: 0.8772 - val_accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 3/200
4/4 - 0s - loss: 0.8585 - accuracy: 0.9444 - val_loss: 0.7869 - val_accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 4/200
4/4 - 0s - loss: 0.7765 - accuracy: 0.8704 - val_loss: 0.6911 - val_accuracy: 0.8333 - 46ms/epoch - 11ms/step
Epoch 5/200
4/4 - 0s - loss: 0.6974 - accuracy: 0.8241 - val_loss: 0.6142 - val_accuracy: 0.8333 - 48ms/epoch - 12ms/step
Epoch 6/200
4/4 - 0s - loss: 0.6284 - accuracy: 0.8056 - val_loss: 0.5381 - val_accuracy: 0.8333 - 38ms/epoch - 10ms/step
Epoch 7/200
4/4 - 0s - loss: 0.5586 - accuracy: 0.8704 - val_loss: 0.4857 - val_accuracy: 1.0000 - 47ms/epoch - 12ms/step
Epoch 8/200
4/4 - 0s - loss: 0.5048 - accuracy: 0.9815 - val_loss: 0.4141 - val_accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 9/200
4/4 - 0s - los

Epoch 68/200
4/4 - 0s - loss: 0.0279 - accuracy: 0.9907 - val_loss: 0.0551 - val_accuracy: 1.0000 - 45ms/epoch - 11ms/step
Epoch 69/200
4/4 - 0s - loss: 0.0450 - accuracy: 0.9815 - val_loss: 0.0409 - val_accuracy: 1.0000 - 40ms/epoch - 10ms/step
Epoch 70/200
4/4 - 0s - loss: 0.0307 - accuracy: 0.9907 - val_loss: 0.0145 - val_accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 71/200
4/4 - 0s - loss: 0.0382 - accuracy: 0.9907 - val_loss: 0.0340 - val_accuracy: 1.0000 - 40ms/epoch - 10ms/step
Epoch 72/200
4/4 - 0s - loss: 0.0312 - accuracy: 1.0000 - val_loss: 0.0786 - val_accuracy: 0.9167 - 41ms/epoch - 10ms/step
Epoch 73/200
4/4 - 0s - loss: 0.0438 - accuracy: 0.9907 - val_loss: 0.0162 - val_accuracy: 1.0000 - 40ms/epoch - 10ms/step
Epoch 74/200
4/4 - 0s - loss: 0.0287 - accuracy: 0.9907 - val_loss: 0.0586 - val_accuracy: 1.0000 - 40ms/epoch - 10ms/step
Epoch 75/200
4/4 - 0s - loss: 0.0263 - accuracy: 1.0000 - val_loss: 0.0137 - val_accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 76/200
4/4 

Epoch 135/200
4/4 - 0s - loss: 0.0121 - accuracy: 1.0000 - val_loss: 0.0118 - val_accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 136/200
4/4 - 0s - loss: 0.0129 - accuracy: 1.0000 - val_loss: 0.0033 - val_accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 137/200
4/4 - 0s - loss: 0.0328 - accuracy: 0.9907 - val_loss: 0.0028 - val_accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 138/200
4/4 - 0s - loss: 0.0103 - accuracy: 1.0000 - val_loss: 0.2221 - val_accuracy: 0.9167 - 38ms/epoch - 9ms/step
Epoch 139/200
4/4 - 0s - loss: 0.0480 - accuracy: 0.9722 - val_loss: 0.0021 - val_accuracy: 1.0000 - 34ms/epoch - 9ms/step
Epoch 140/200
4/4 - 0s - loss: 0.0390 - accuracy: 0.9907 - val_loss: 0.0034 - val_accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 141/200
4/4 - 0s - loss: 0.0253 - accuracy: 0.9907 - val_loss: 0.0594 - val_accuracy: 0.9167 - 36ms/epoch - 9ms/step
Epoch 142/200
4/4 - 0s - loss: 0.0603 - accuracy: 0.9722 - val_loss: 0.0029 - val_accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 143/200
4/